# Module 3 Lab — Standards, Regulation & Governance Operating Model

**Scenario:** Enterprise Procurement Agent  
**Purpose:** Turn frameworks and regulatory inputs into a repeatable governance workflow.

You will implement:

```text
System Inventory
   ↓
Applicability Profile
   ↓
Risk / Autonomy Context
   ↓
Control Selection
   ↓
RACI
   ↓
Evidence Requirements
   ↓
Stage-Gate Decision
   ↓
Exception / Approval
   ↓
Change Detection
   ↓
Recertification
```

> This lab is educational and technical, not legal advice.

## Current tools

- **Pydantic** — typed governance records
- **pandas** — control crosswalks, dashboards, evidence matrices
- **jsonschema** — machine-readable artifact validation
- **NetworkX** — responsibility/dependency checks
- **Jinja2** — governance evidence/approval pack
- **Compliance Trestle / OSCAL** — optional compliance-as-code workflow
- **OpenAI SDK** — optional structured extraction from architecture descriptions

In [ ]:
%pip install -q "pydantic>=2" pandas networkx jsonschema jinja2 openai
print("Core dependencies installed.")

In [ ]:
from __future__ import annotations
from datetime import date, datetime, timezone, timedelta
from enum import Enum
from typing import Optional, Any
from uuid import uuid4
import json, os, hashlib

import pandas as pd
import networkx as nx
from pydantic import BaseModel, Field
from jsonschema import validate
from jinja2 import Template

pd.set_option("display.max_colwidth", 120)

## 1. Structured AI / Agent System Inventory

In [ ]:
class GovernanceStatus(str, Enum):
    DRAFT = "draft"
    ASSESSMENT = "assessment"
    CONDITIONALLY_APPROVED = "conditionally_approved"
    APPROVED = "approved"
    REJECTED = "rejected"
    SUSPENDED = "suspended"

class AgentSystemRecord(BaseModel):
    system_id: str = Field(default_factory=lambda: f"ai-{uuid4().hex[:8]}")
    name: str
    purpose: str
    business_owner: str
    technical_owner: str
    governance_owner: str
    autonomy_level: str
    internal_risk_tier: str
    models: list[str]
    tools: list[str]
    data_classes: list[str]
    memory_enabled: bool = False
    subagents_enabled: bool = False
    jurisdictions: list[str]
    status: GovernanceStatus = GovernanceStatus.DRAFT
    version: str = "1.0.0"
    next_review: date
    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

procurement = AgentSystemRecord(
    name="Enterprise Procurement Agent",
    purpose="Support routine purchasing while preserving financial and vendor controls.",
    business_owner="VP Procurement",
    technical_owner="AI Platform Lead",
    governance_owner="Responsible AI Office",
    autonomy_level="bounded_autonomy",
    internal_risk_tier="high",
    models=["enterprise-llm"],
    tools=["vendor_search","supplier_email","create_purchase_order"],
    data_classes=["internal","confidential"],
    memory_enabled=True,
    subagents_enabled=True,
    jurisdictions=["CA","EU"],
    next_review=date.today()+timedelta(days=180),
)

display(pd.DataFrame([procurement.model_dump()]))

### Why the inventory matters

Governance begins with a system of record.

It should answer:

- Who owns this?
- What is it for?
- What can it do?
- What can it access?
- Where does it operate?
- What risk tier applies?
- When was it approved?
- When must it be reviewed again?

## 2. Separate internal risk from regulatory applicability

In [ ]:
class EUAIActProfile(BaseModel):
    in_scope: bool
    organization_role: str
    possible_high_risk_review: bool
    transparency_review: bool
    gpai_provider: bool
    notes: list[str] = []

def eu_ai_act_training_screen(record: AgentSystemRecord) -> EUAIActProfile:
    # TRAINING APPROXIMATION ONLY.
    # It is intentionally conservative and does not replace legal analysis.
    in_eu = "EU" in record.jurisdictions
    notes = []
    if not in_eu:
        return EUAIActProfile(
            in_scope=False,
            organization_role="unknown/not assessed",
            possible_high_risk_review=False,
            transparency_review=False,
            gpai_provider=False,
            notes=["No EU deployment declared in system inventory."],
        )

    notes.append("EU deployment declared: perform formal role and AI Act applicability review.")
    if record.autonomy_level in {"bounded_autonomy","high_autonomy"}:
        notes.append("Autonomy does not itself determine AI Act high-risk classification.")
    return EUAIActProfile(
        in_scope=True,
        organization_role="deployer (training assumption)",
        possible_high_risk_review=True,
        transparency_review=True,
        gpai_provider=False,
        notes=notes,
    )

eu_profile = eu_ai_act_training_screen(procurement)
print(eu_profile.model_dump_json(indent=2))

### Important

Do **not** encode legal interpretation into a simplistic risk score.

Store:

```text
Internal engineering risk tier
Regulatory applicability profile
Sector obligations
Jurisdiction
```

separately.

A legal/compliance specialist can then confirm the regulatory assessment.

## 3. Build the internal control library

In [ ]:
class Control(BaseModel):
    control_id: str
    title: str
    objective: str
    applies_when: list[str]
    evidence_required: list[str]
    owner_role: str
    mappings: dict[str, list[str]]

controls = [
    Control(
        control_id="AG-INV-001",
        title="AI system inventory",
        objective="Maintain a current system record, purpose, owner, risk tier, architecture, and lifecycle status.",
        applies_when=["all_ai_systems"],
        evidence_required=["inventory_record","owner_attestation"],
        owner_role="AI Governance",
        mappings={
            "NIST_AI_RMF":["GOVERN","MAP"],
            "ISO_42001":["AIMS governance/context"],
            "EU_AI_Act":["documentation/applicability support"],
        },
    ),
    Control(
        control_id="AG-RSK-001",
        title="Agent risk assessment",
        objective="Assess autonomy, impact, access, irreversibility, uncertainty, and residual risk.",
        applies_when=["agentic_system"],
        evidence_required=["risk_assessment","risk_owner_acceptance"],
        owner_role="Business Owner",
        mappings={
            "NIST_AI_RMF":["MAP","MEASURE","MANAGE"],
            "ISO_42001":["AI risk management"],
            "ISO_42005":["impact-assessment linkage"],
        },
    ),
    Control(
        control_id="AG-AUTH-001",
        title="Runtime authorization for state-changing actions",
        objective="Prevent execution outside delegated authority.",
        applies_when=["state_changing_tools"],
        evidence_required=["policy_configuration","authorization_tests","denied_action_trace"],
        owner_role="Security Architecture",
        mappings={
            "NIST_AI_RMF":["MANAGE"],
            "ISO_42001":["operational control"],
            "OWASP_AGENTIC":["identity/privilege/tool misuse"],
        },
    ),
    Control(
        control_id="AG-HUM-001",
        title="Risk-based human oversight",
        objective="Escalate material or irreversible actions to appropriately authorized humans.",
        applies_when=["high_impact_action"],
        evidence_required=["approval_policy","approval_test","approval_event"],
        owner_role="Business Owner",
        mappings={
            "NIST_AI_RMF":["GOVERN","MANAGE"],
            "ISO_42001":["human oversight / operations"],
            "EU_AI_Act":["human oversight where applicable"],
        },
    ),
    Control(
        control_id="AG-OBS-001",
        title="Governance traceability",
        objective="Reconstruct goals, tools, policy decisions, approvals, and outcomes.",
        applies_when=["agentic_system"],
        evidence_required=["trace_sample","retention_policy","monitoring_dashboard"],
        owner_role="AI Platform",
        mappings={
            "NIST_AI_RMF":["MEASURE","MANAGE"],
            "ISO_42001":["monitoring/evidence"],
            "EU_AI_Act":["record keeping where applicable"],
        },
    ),
]
control_df = pd.DataFrame([{
    **c.model_dump(exclude={"mappings"}),
    "mappings": json.dumps(c.mappings)
} for c in controls])
display(control_df)

## 4. Select controls based on system profile

In [ ]:
def select_controls(record: AgentSystemRecord, library: list[Control]) -> list[Control]:
    tags={"all_ai_systems"}
    if "autonomy" in record.autonomy_level or record.autonomy_level in {"assisted_execution","bounded_autonomy","high_autonomy"}:
        tags.add("agentic_system")
    if any(t in record.tools for t in ["create_purchase_order","issue_payment","supplier_email"]):
        tags.add("state_changing_tools")
    if record.internal_risk_tier in {"high","critical"}:
        tags.add("high_impact_action")

    selected=[]
    for c in library:
        if any(tag in tags for tag in c.applies_when):
            selected.append(c)
    return selected

selected = select_controls(procurement, controls)
display(pd.DataFrame([{"control_id":c.control_id,"title":c.title,"owner":c.owner_role} for c in selected]))

## 5. Crosswalk: one control, multiple requirements

In [ ]:
crosswalk_rows=[]
for c in selected:
    for framework,refs in c.mappings.items():
        for ref in refs:
            crosswalk_rows.append({
                "control_id":c.control_id,
                "control":c.title,
                "framework":framework,
                "mapped_requirement_or_theme":ref,
            })
crosswalk=pd.DataFrame(crosswalk_rows)
display(crosswalk)

### Caution

A crosswalk means:

> “This internal control contributes evidence relevant to this requirement/theme.”

It does **not** mean:

> “This control automatically proves legal compliance.”

## 6. RACI

In [ ]:
raci = pd.DataFrame([
    ["Define purpose","A/R","C","C","I","C","I"],
    ["Architecture","C","A/R","C","C","I","I"],
    ["Risk classification","A","C","R","C","C","I"],
    ["Threat modeling","I","C","C","A/R","I","I"],
    ["Regulatory applicability","C","I","C","I","A/R","I"],
    ["Production approval","A","C","R","C","C","I"],
    ["Runtime monitoring","A","R","C","C","I","I"],
    ["Independent assurance","I","I","I","I","I","A/R"],
],columns=["activity","Business Owner","AI Engineering","AI Governance","Security","Legal/Privacy","Audit"])
display(raci)

## 7. Responsibility graph — detect ownership gaps

In [ ]:
R = nx.DiGraph()
activities = list(raci["activity"])
roles = list(raci.columns[1:])

for activity in activities:
    R.add_node(activity, kind="activity")

for role in roles:
    R.add_node(role, kind="role")

for _,row in raci.iterrows():
    for role in roles:
        if "A" in row[role]:
            R.add_edge(role,row["activity"],relation="accountable")
        if "R" in row[role]:
            R.add_edge(role,row["activity"],relation="responsible")

gaps=[]
for activity in activities:
    accountable=[u for u,v,d in R.in_edges(activity,data=True) if d["relation"]=="accountable"]
    responsible=[u for u,v,d in R.in_edges(activity,data=True) if d["relation"]=="responsible"]
    if len(accountable)!=1 or len(responsible)<1:
        gaps.append({"activity":activity,"accountable":accountable,"responsible":responsible})

print("RACI gaps:", gaps or "None")

## 8. Evidence registry

In [ ]:
class EvidenceItem(BaseModel):
    evidence_id: str = Field(default_factory=lambda: f"ev-{uuid4().hex[:8]}")
    control_id: str
    evidence_type: str
    artifact_uri: str
    version: str
    collected_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    valid_until: Optional[date] = None
    status: str = "valid"

evidence = [
    EvidenceItem(control_id="AG-INV-001", evidence_type="inventory_record", artifact_uri="governance://systems/procurement-agent", version="1.0"),
    EvidenceItem(control_id="AG-RSK-001", evidence_type="risk_assessment", artifact_uri="repo://risk/module02_agent_risk_register.csv", version="1.0"),
    EvidenceItem(control_id="AG-AUTH-001", evidence_type="authorization_tests", artifact_uri="ci://agent-auth/tests/245", version="1.3"),
    EvidenceItem(control_id="AG-OBS-001", evidence_type="trace_sample", artifact_uri="otel://traces/procurement-agent", version="1.0"),
]

evidence_df=pd.DataFrame([e.model_dump() for e in evidence])
display(evidence_df)

## 9. Evidence completeness

In [ ]:
def evidence_completeness(selected_controls: list[Control], evidence: list[EvidenceItem]) -> pd.DataFrame:
    have={(e.control_id,e.evidence_type) for e in evidence if e.status=="valid"}
    rows=[]
    for c in selected_controls:
        required=set(c.evidence_required)
        present={etype for cid,etype in have if cid==c.control_id}
        missing=required-present
        rows.append({
            "control_id":c.control_id,
            "control":c.title,
            "required":len(required),
            "present":len(required & present),
            "completeness":round(len(required & present)/len(required),2) if required else 1.0,
            "missing":", ".join(sorted(missing)),
        })
    return pd.DataFrame(rows)

completeness=evidence_completeness(selected,evidence)
display(completeness)

## 10. Governance stage-gate decision

In [ ]:
class GateDecision(str, Enum):
    APPROVE = "APPROVE"
    APPROVE_WITH_CONDITIONS = "APPROVE_WITH_CONDITIONS"
    REJECT = "REJECT"
    EXCEPTION_REQUIRED = "EXCEPTION_REQUIRED"

def gate_decision(record: AgentSystemRecord, completeness: pd.DataFrame) -> tuple[GateDecision,list[str]]:
    reasons=[]
    avg=float(completeness["completeness"].mean())
    critical_missing = completeness[
        (completeness["control_id"].isin(["AG-AUTH-001","AG-RSK-001"]))
        & (completeness["completeness"] < 1)
    ]

    if len(critical_missing):
        reasons.append("Critical authorization/risk evidence is incomplete.")
        return GateDecision.REJECT,reasons
    if avg < 0.85:
        reasons.append(f"Evidence completeness is only {avg:.0%}.")
        return GateDecision.EXCEPTION_REQUIRED,reasons
    if avg < 1.0:
        reasons.append("Non-critical evidence remains incomplete.")
        return GateDecision.APPROVE_WITH_CONDITIONS,reasons
    reasons.append("Required evidence is complete.")
    return GateDecision.APPROVE,reasons

decision,reasons=gate_decision(procurement,completeness)
print(decision.value,reasons)

## 11. Exceptions must expire

In [ ]:
class ExceptionRecord(BaseModel):
    exception_id: str = Field(default_factory=lambda: f"ex-{uuid4().hex[:8]}")
    system_id: str
    control_id: str
    rationale: str
    residual_risk: str
    compensating_controls: list[str]
    risk_acceptor: str
    expires_on: date
    remediation_plan: str
    status: str = "open"

exception = ExceptionRecord(
    system_id=procurement.system_id,
    control_id="AG-HUM-001",
    rationale="Approval workflow integration delayed.",
    residual_risk="high-impact actions could lack standard approval orchestration",
    compensating_controls=["temporarily disable autonomous PO creation"],
    risk_acceptor="VP Procurement",
    expires_on=date.today()+timedelta(days=30),
    remediation_plan="Deploy approval service integration before expiration.",
)
print(exception.model_dump_json(indent=2))

## 12. Material-change detector

In [ ]:
class SystemSnapshot(BaseModel):
    version: str
    models: set[str]
    tools: set[str]
    data_classes: set[str]
    autonomy_level: str
    memory_enabled: bool
    subagents_enabled: bool
    jurisdictions: set[str]
    max_transaction: float

old = SystemSnapshot(
    version="1.0.0",
    models={"enterprise-llm-v1"},
    tools={"vendor_search","supplier_email"},
    data_classes={"internal"},
    autonomy_level="assisted_execution",
    memory_enabled=False,
    subagents_enabled=False,
    jurisdictions={"CA"},
    max_transaction=5_000,
)

new = SystemSnapshot(
    version="1.1.0",
    models={"enterprise-llm-v2"},
    tools={"vendor_search","supplier_email","create_purchase_order"},
    data_classes={"internal","confidential"},
    autonomy_level="bounded_autonomy",
    memory_enabled=True,
    subagents_enabled=True,
    jurisdictions={"CA","EU"},
    max_transaction=25_000,
)

def material_changes(a:SystemSnapshot,b:SystemSnapshot):
    changes=[]
    if a.models != b.models: changes.append("model_change")
    if b.tools-a.tools: changes.append(f"new_tools:{sorted(b.tools-a.tools)}")
    if b.data_classes-a.data_classes: changes.append(f"new_data_classes:{sorted(b.data_classes-a.data_classes)}")
    if a.autonomy_level != b.autonomy_level: changes.append("autonomy_change")
    if not a.memory_enabled and b.memory_enabled: changes.append("memory_enabled")
    if not a.subagents_enabled and b.subagents_enabled: changes.append("subagents_enabled")
    if b.jurisdictions-a.jurisdictions: changes.append(f"new_jurisdictions:{sorted(b.jurisdictions-a.jurisdictions)}")
    if b.max_transaction > a.max_transaction: changes.append("higher_transaction_limit")
    return changes

changes=material_changes(old,new)
changes

In [ ]:
def reassessment_scope(changes:list[str]) -> list[str]:
    scope=set()
    for change in changes:
        if change=="model_change":
            scope.update(["model evaluation","safety evaluation","cost/latency regression"])
        if change.startswith("new_tools"):
            scope.update(["authorization","threat model","tool governance","runtime policy"])
        if change.startswith("new_data_classes"):
            scope.update(["privacy","data governance","access control"])
        if change=="autonomy_change":
            scope.update(["agent risk assessment","human oversight","approval architecture"])
        if change=="memory_enabled":
            scope.update(["memory governance","retention","data isolation"])
        if change=="subagents_enabled":
            scope.update(["delegation risk","agent identity","authority chain"])
        if change.startswith("new_jurisdictions"):
            scope.update(["legal/regulatory applicability"])
        if change=="higher_transaction_limit":
            scope.update(["financial risk","approval thresholds","risk appetite"])
    return sorted(scope)

reassessment_scope(changes)

## 13. Machine-readable governance artifact

In [ ]:
governance_artifact = {
    "schema_version":"oneplusi-governance-1.0",
    "system":procurement.model_dump(mode="json"),
    "eu_ai_act_training_profile":eu_profile.model_dump(mode="json"),
    "selected_controls":[c.model_dump(mode="json") for c in selected],
    "evidence":[e.model_dump(mode="json") for e in evidence],
    "gate":{
        "decision":decision.value,
        "reasons":reasons,
    },
    "generated_at":datetime.now(timezone.utc).isoformat(),
}

with open("procurement_governance_artifact.json","w") as f:
    json.dump(governance_artifact,f,indent=2)

print("Saved procurement_governance_artifact.json")

## 14. Validate the governance artifact with JSON Schema

In [ ]:
artifact_schema = {
    "type":"object",
    "required":["schema_version","system","selected_controls","evidence","gate"],
    "properties":{
        "schema_version":{"type":"string"},
        "system":{
            "type":"object",
            "required":["system_id","name","business_owner","technical_owner","autonomy_level","internal_risk_tier"]
        },
        "selected_controls":{"type":"array","minItems":1},
        "evidence":{"type":"array"},
        "gate":{
            "type":"object",
            "required":["decision","reasons"]
        }
    }
}

validate(instance=governance_artifact,schema=artifact_schema)
print("Governance artifact validated.")

# 15. OSCAL / Compliance Trestle — optional compliance-as-code extension

NIST OSCAL provides machine-readable models for control catalogs, system implementation, and assessment information.

**Compliance Trestle** provides Git/CI-oriented workflows around OSCAL.

Install separately if you want to experiment:

```bash
pip install compliance-trestle
trestle init
```

Then use Trestle to manage a valid OSCAL catalog/profile or system-security-plan artifact.

Project:
https://github.com/oscal-compass/compliance-trestle

NIST OSCAL:
https://pages.nist.gov/OSCAL/

### Why this is optional here

The governance artifact above contains AI-agent-specific metadata that does not map one-to-one to OSCAL security models. In a real platform, use a mapping layer:

```text
AI System Registry
      ↓
Internal Control Library
      ↔
OSCAL control/evidence artifacts
      ↓
GRC / assurance / audit tooling
```

Do not claim OSCAL conformance unless the artifact validates against the official OSCAL model/schema.

## 16. Generate a governance approval pack

In [ ]:
template = Template('''
# Governance Decision — {{ system.name }}

**System ID:** {{ system.system_id }}
**Purpose:** {{ system.purpose }}
**Business owner:** {{ system.business_owner }}
**Technical owner:** {{ system.technical_owner }}
**Autonomy:** {{ system.autonomy_level }}
**Internal risk tier:** {{ system.internal_risk_tier }}

## Regulatory screening
EU AI Act training screen: {{ eu.in_scope }}
Possible high-risk review: {{ eu.possible_high_risk_review }}

## Decision
**{{ gate.decision }}**

{% for reason in gate.reasons -%}
- {{ reason }}
{% endfor %}

## Required controls
{% for c in controls -%}
- {{ c.control_id }} — {{ c.title }}
{% endfor %}

## Evidence completeness
{{ evidence_summary }}

## Material-change triggers
Model, tool, data, autonomy, memory, delegation, jurisdiction, and material transaction-limit changes require reassessment.
''')

approval_pack = template.render(
    system=procurement,
    eu=eu_profile,
    gate={"decision":decision.value,"reasons":reasons},
    controls=selected,
    evidence_summary=completeness[["control_id","completeness","missing"]].to_markdown(index=False),
)

with open("procurement_governance_decision.md","w") as f:
    f.write(approval_pack)

print(approval_pack)

## 17. Governance dashboard

In [ ]:
dashboard = {
    "registered_systems":1,
    "systems_with_named_owner":1,
    "high_risk_systems":1 if procurement.internal_risk_tier=="high" else 0,
    "evidence_completeness":round(float(completeness["completeness"].mean()),2),
    "open_exceptions":1,
    "exceptions_expiring_30d":1 if exception.expires_on <= date.today()+timedelta(days=30) else 0,
    "next_review_days":(procurement.next_review-date.today()).days,
}
display(pd.DataFrame([dashboard]).T.rename(columns={0:"value"}))

# 18. Optional LLM-assisted artifact extraction

An LLM can help extract candidate inventory fields from architecture documentation.

Use it as an **assistant**, not the authoritative governance source.

Pattern:

```text
Architecture / README
   ↓
LLM structured extraction
   ↓
Pydantic validation
   ↓
Human/system-owner confirmation
   ↓
Governance system of record
```

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    from openai import OpenAI
    client=OpenAI()

    architecture_text = '''
    Procurement Agent supports Canadian and EU users. It searches approved vendors,
    emails suppliers, stores vendor preferences, and creates purchase orders up to
    $25,000. A procurement manager owns the business process. The AI Platform team
    operates the service.
    '''

    class ExtractedInventory(BaseModel):
        purpose: str
        jurisdictions: list[str]
        tools: list[str]
        memory_enabled: bool
        max_transaction: Optional[float]
        questions_for_owner: list[str]

    response = client.responses.parse(
        model=os.getenv("OPENAI_MODEL","gpt-5.6-sol"),
        input=[
            {"role":"system","content":"Extract candidate AI governance inventory fields. Flag unknowns; do not invent legal classifications."},
            {"role":"user","content":architecture_text},
        ],
        text_format=ExtractedInventory,
    )
    print(response.output_parsed.model_dump_json(indent=2))
else:
    print("Set OPENAI_API_KEY to run the optional structured extraction.")

# 19. Governance regression tests

Governance artifacts should be testable.

Examples:

- every production system has one accountable business owner,
- every state-changing agent has authorization controls,
- every high-risk system has a risk assessment,
- exceptions have expiration dates,
- evidence completeness meets the release threshold.

In [ ]:
assert procurement.business_owner
assert procurement.technical_owner
assert procurement.next_review > date.today()

selected_ids={c.control_id for c in selected}
assert "AG-RSK-001" in selected_ids
assert "AG-AUTH-001" in selected_ids

assert exception.expires_on > date.today()
assert exception.risk_acceptor

print("Governance regression tests passed.")

# 20. Exercises

### A. Add a second system

Create a low-risk internal summarization assistant.

Compare:

- controls,
- RACI,
- stage gates,
- evidence burden.

### B. Add a payment tool

Update the Procurement Agent and run:

- control selection,
- material-change detection,
- reassessment scope.

### C. Add a US deployment

Extend jurisdiction fields and define a process for specialist regulatory review without hard-coding legal conclusions.

### D. Build an impact assessment

Create an ISO 42005-inspired impact record for:

- employees,
- suppliers,
- procurement staff,
- customers indirectly affected.

### E. Compliance-as-code

Install `compliance-trestle`, initialize a workspace, and explore how an internal control catalog could be represented in OSCAL.

### F. Exception expiry

Make the exception expire yesterday. Write a governance test that fails the pipeline.

### G. RACI conflict

Assign two accountable owners to one activity. Detect the governance defect programmatically.

# 21. Key takeaways

1. A framework is not an operating model.
2. Separate internal risk tier from regulatory applicability.
3. NIST AI RMF, ISO 42001, ISO 42005, regulation, and security guidance solve different layers.
4. An inventory is foundational.
5. Controls should have stable IDs, owners, tests, and evidence.
6. Crosswalks reduce duplication but do not automatically establish compliance.
7. Governance gates should be risk-based.
8. Business/system owners remain accountable.
9. Exceptions need expiry and remediation.
10. Material change should trigger reassessment.
11. Evidence should be produced continuously.
12. Machine-readable governance and OSCAL-style workflows can improve scale and auditability.